# Lorenz-63 Model

The Lorenz-63 model is commonly used to assess various data assimilation methods for nonlinear systems. This model is of interest because of its nonlinear chaotic behavior and low dimensions. The simple three-variable model is composed of a system of nonlinear ordinary differential equations:
\begin{gather*}
	\frac{\partial x}{\partial t} &=& \sigma (y-x) \\
	\frac{\partial y}{\partial t} &=& \rho x-y-xz \\
	\frac{\partial z}{\partial t} &=& xy-\beta z
\end{gather*}
We will use the following common parameter values for the reference solution: $\sigma = 10$, $\rho = 28$, and $\beta = 8/3$. 

<!-- The reference solution will have the initial conditions of $(x_0, y_0, z_0) = (1.508870, -1.531271, 25.46091)$ and a time interval of $t \in [0, 40]$. -->

The observations are generated by adding Gaussian noise with zero mean and a variance of 4.0 (standard deviation of 2.0) to the reference solution and will be measured at regular time intervals $\Delta t = 0.1$.

In [1]:
# LIBRARIES

from utils.plots import plot_rmses, plot_timeplots, plot_particles, plot_sigmas

import os, glob, numpy as np

# import ipywidgets
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

In [2]:
def create_jobs():
    save_path = "./slurms/"
    files = glob.glob(os.path.join(save_path, "slurm-*.out"))

    for file_path in files:
        os.remove(file_path)

    ensembles = [10, 20, 40, 60, 100, 200, 400, 600]

    filter_variants = [
        {"type": "KDE", "scheduler": "VE", "sigma_max": 50},
        {"type": "KDE", "scheduler": "VP", "sigma_max": 10},
        {"type": "KDE_hybrid", "scheduler": "VE", "sigma_max": 50},
        {"type": "KDE_hybrid", "scheduler": "VP", "sigma_max": 10},
        {"type": "EnKF"}
    ] 

    obs_ops = ['linear', 'cubic']

    sigma_min_xs = [0.1, 0.2, 0.3, 0.4]
    sigma_ys = [1, 2]

    i = 1
    for ensemble in ensembles:
        for filter_config in filter_variants:
            filter_type = filter_config["type"]

            if filter_type == "KDE":
                for obs_op in obs_ops:
                    for sigma_min_x in sigma_min_xs:
                        for sigma_y in sigma_ys:
                            scheduler = filter_config["scheduler"]
                            sigma_max = filter_config["sigma_max"]

                            f = open(save_path + 'train'+str(i)+'.slurm', 'w')
                            print('#!/bin/bash' ,file=f)
                            print('#SBATCH --partition=epyc-64' ,file=f)
                            print('#SBATCH --nodes=1' ,file=f)
                            print('#SBATCH --ntasks=1' ,file=f)
                            print('#SBATCH --cpus-per-task=4' ,file=f)
                            print('#SBATCH --mem=48GB' ,file=f)
                            print('#SBATCH --time=24:00:00' ,file=f)

                            print("",file=f)
                            print('eval "$(conda shell.bash hook)"', file=f)
                            print('conda activate da', file=f)

                            print("",file=f)
                            print(f"python3 lorenz63_setup.py --ensemble {ensemble} --obs_op_type {obs_op} --filter_type KDE --scheduler {scheduler} --sigma_min_x {sigma_min_x} --sigma_max_x {sigma_max} --sigma_y {sigma_y}",file=f)
                            print("",file=f)
                            f.close()
                            i = i+1

            elif filter_type == "KDE_hybrid":
                for sigma_min_x in sigma_min_xs:
                    for sigma_y in sigma_ys:
                        scheduler = filter_config["scheduler"]
                        sigma_max = filter_config["sigma_max"]

                        f = open(save_path + 'train'+str(i)+'.slurm', 'w')
                        print('#!/bin/bash' ,file=f)
                        print('#SBATCH --partition=epyc-64' ,file=f)
                        print('#SBATCH --nodes=1' ,file=f)
                        print('#SBATCH --ntasks=1' ,file=f)
                        print('#SBATCH --cpus-per-task=4' ,file=f)
                        print('#SBATCH --mem=48GB' ,file=f)
                        print('#SBATCH --time=24:00:00' ,file=f)

                        print("",file=f)
                        print('eval "$(conda shell.bash hook)"', file=f)
                        print('conda activate da', file=f)

                        print("",file=f)
                        print(f"python3 lorenz63_setup.py --ensemble {ensemble} --filter_type KDE_hybrid --scheduler {scheduler} --sigma_min_x {sigma_min_x} --sigma_max_x {sigma_max} --sigma_y {sigma_y}",file=f)
                        print("",file=f)
                        f.close()
                        i = i+1

            elif filter_type == 'EnKF':
                f = open(save_path + 'train'+str(i)+'.slurm', 'w')
                print('#!/bin/bash' ,file=f)
                print('#SBATCH --partition=epyc-64' ,file=f)
                print('#SBATCH --nodes=1' ,file=f)
                print('#SBATCH --ntasks=1' ,file=f)
                print('#SBATCH --cpus-per-task=4' ,file=f)
                print('#SBATCH --mem=48GB' ,file=f)
                print('#SBATCH --time=24:00:00' ,file=f)

                print("",file=f)
                print('eval "$(conda shell.bash hook)"', file=f)
                print('conda activate da', file=f)

                print("",file=f)
                print(f"python3 lorenz63_setup.py --ensemble {ensemble} --filter_type EnKF",file=f)
                print("",file=f)
                f.close()
                i = i+1

    print("Done")

create_jobs()

Done


In [3]:
# plot_rmses(linear_models)

In [4]:
# plot_particles(linear_models)

In [5]:
# plot_timeplots(linear_models)